# Notebook 01: UCI OPPORTUNITY Dataset Exploration & Verification

**Objective**: Inspect downloaded raw files, column mappings, sensory modalities, and activity labels without making assumptions or modifying raw files.

> **Research Rule**: Never fabricate column headers, sensor mappings, or sampling frequency. Derive all metadata directly from official documentation.

## 0. Environment Setup & Repository Bootstrap

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT DETECTION & REPOSITORY BOOTSTRAP
# ==============================================================================
import sys
import os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/Opportunity_HAR")
    if not REPO_DIR.exists():
        print("[INFO] Cloning Opportunity_HAR repository from GitHub...")
        !git clone https://github.com/DhruvikaRajput/Opportunity_HAR.git {REPO_DIR}
    else:
        print(f"[INFO] Repository already present at {REPO_DIR}. Pulling latest changes...")
        !git -C {REPO_DIR} pull origin main

    # Set repository root at front of sys.path
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
else:
    current = Path.cwd()
    REPO_DIR = current.parent if current.name == "notebooks" else current
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    os.chdir(REPO_DIR)

print(f"Active Working Directory : {Path.cwd()}")

# Ensure fresh import of src.api
import importlib
if "src.api" in sys.modules:
    import src.api
    importlib.reload(src.api)
else:
    import src.api as api

import src.api as api
status = api.setup_project()
print(f"High-Level API Loaded    : src.api (locate_dataset available: {hasattr(api, 'locate_dataset')})")
print(f"Compute Device Detected  : {status['device_name']} (CUDA Available: {status['cuda_available']})")

## 1. Locate Dataset & Run Automated Audit
Audits the 24 raw `.dat` files, parses official `column_names.txt`, and generates machine-readable metadata in `data/processed/`.

In [ ]:
ds_status = api.locate_dataset()
print(ds_status["message"])

if ds_status["found"]:
    from src.data.inspect_opportunity import run_inspection
    report = run_inspection()
    print(f"Total Files Inspected : {report['total_files']}")
    print(f"Total Timesteps       : {report['total_rows']:,}")
    print(f"Columns per File      : {report['columns_per_file']}")
    print(f"Sampling Frequency    : {report['sampling_rate_hz']} Hz")
    print(f"On-Body Channels      : {len(report['sensor_categories']['on_body_sensors'])} channels")
else:
    print("\n[ACTION NEEDED] Please mount Google Drive and ensure your OPPORTUNITY .dat files are placed in data/raw/ or Drive.")

## 2. Inspect a Real Continuous Recording (Subject 1, ADL 1)

In [ ]:
if ds_status["found"]:
    data, labels, timestamps = api.load_opportunity_data(subject="S1", run="ADL1", target_track="Locomotion")
    print(f"Recording Shape: {data.shape} (timesteps, on-body channels)")
    print(f"Labels Count   : {len(labels)}")
    print(f"Sampling Rate  : {1000.0 / np.median(np.diff(timestamps)):.1f} Hz")
    import pandas as pd
    print(f"Label Counts   : {pd.Series(labels).value_counts().to_dict()}")
else:
    print("Dataset not located yet; skipping recording inspection.")

## 3. Visualize Multi-Sensor Inertial Waveforms

In [ ]:
if ds_status["found"]:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
    time_range = slice(1000, 1600)  # ~20 seconds of activity

    axes[0].plot(data[time_range, 0:3], label=["Back_Acc_X", "Back_Acc_Y", "Back_Acc_Z"])
    axes[0].set_title("Trunk Acceleration")
    axes[0].legend(loc="upper right", fontsize=8)

    axes[1].plot(data[time_range, 25:28], label=["R_Arm_Acc_X", "R_Arm_Acc_Y", "R_Arm_Acc_Z"])
    axes[1].set_title("Right Arm Acceleration")
    axes[1].legend(loc="upper right", fontsize=8)

    axes[2].plot(labels[time_range], color="red", label="Locomotion Label")
    axes[2].set_title("Locomotion Ground Truth (Stand, Walk, Sit, Lie)")
    axes[2].set_xlabel("Timestep (30 Hz)")
    axes[2].legend(loc="upper right", fontsize=8)

    plt.tight_layout()
    plt.show()
else:
    print("Dataset not located yet; skipping visualization.")